# Formal Run Notebook

- Runtime version: 2026.04
- GPU: NVIDIA A100-SXM4-80GB

In [1]:
# re-run on session restart (lightweight dependencies)
from google.colab import userdata
import os
import json
from huggingface_hub import login

GITHUB_PAT, HF_PAT = userdata.get("GH_PAT"), userdata.get("HF_TOKEN-cl-prm-project")

login(HF_PAT)

# Stage 1: Setup

- clone latest repo `main` branch
- install heavy-dependencies

In [2]:
# git remote set-url origin repo
!git clone https://{GITHUB_PAT}@github.com/Zhiheng-SC/cl-prm-project

Cloning into 'cl-prm-project'...
remote: Enumerating objects: 1036, done.
remote: Counting objects: 100% (59/59), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 1036 (delta 0), reused 0 (delta 0), pack-reused 977 (from 2)
Receiving objects: 100% (1036/1036), 1.28 MiB | 15.74 MiB/s, done.
Resolving deltas: 100% (435/435), done.


In [2]:
# re-run on session restart
%cd cl-prm-project
!ls

/content/cl-prm-project
configs  experiments	 README.md		  requirements.txt  tests
data	 outputs	 requirements-common.txt  scripts
docs	 pyproject.toml  requirements-runpod.txt  src


* session restarts after this cell run
* you will need to re-run the `first` and `third` cells

In [4]:
!pip install torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu128
!pip install flash-attn==2.8.3.post1
!pip install -r requirements-runpod.txt

Looking in indexes: https://download.pytorch.org/whl/cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 159.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 136.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 MB 19.1 MB/s eta 0:00:00
  Attempting uninstall: triton
    Found existing installation: triton 3.6.0
    Uninstalling triton-3.6.0:
      Successfully uninstalled triton-3.6.0
  Attempting uninstall: nvidia-nccl-cu12
    Found existing installation: nvidia-nccl-cu12 2.27.5
    Uninstalling nvidia-nccl-cu12-2.27.5:
      Successfully uninstalled nvidia-nccl-cu12-2.27.5
  Attempting uninstall: torch
    Found existing installation: torch 2.10.0+cu128
    Uninstalling torch-2.10.0+cu128:
      Successfully uninstalled torch-2.10.0+cu128
  Attempting uninstall: tor

### Build and Verify Environment

In [3]:
!bash scripts/bootstrap_runpod.sh

Checking the image-provided CUDA binary stack...
Python: 3.12.13
Torch: 2.8.0+cu128
Torch CUDA: 12.8
FlashAttention: 2.8.3.post1
GPU: NVIDIA A100-SXM4-80GB
Kernel: (1, 128, 4, 64) torch.bfloat16 cuda:0
Installing RunPod-safe project dependencies...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 59.4 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
Obtaining file:///content/cl-prm-project
  Checking if build backend supports build_editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for cl-prm (pyproject.toml) ... done
  Created wheel for cl-prm: filename=cl_prm-0.1.0-0.editable-py3-none-any.whl size=1204 sha256=d2a97adcb36612421bbbf74617712c66e490ba4c4be7475fc5cce22e0c50ef2d
  Stored in directory: /tmp/pip-ephem-wheel-cache-knvzkvfv/wheels/4d/6e/d5/e7b30470ebf1f41dc1fbe9a1ba00e4f3ea58b215925e01810e
Successfully built cl-prm
Rech

### Prepare Formal Splits

In [4]:
!python experiments/feasibility/prm_router/prepare_subset.py \
    --n-correct 50 \
    --n-error 50 \
    --seed 2026 \
    --output data/prm_router/feasibility_100.jsonl

Loading dataset: hitsmy/PRMBench_Preview
Dataset revision: 5cc7683d0ae5797f84d7aeac0607966f277c39e1
README.md: 3.52kB [00:00, 692kB/s]
prmbench_preview.jsonl: 100% 20.4M/20.4M [00:00<00:00, 21.3MB/s]
Generating train split: 6216 examples [00:00, 53355.94 examples/s]
Using split: train
Number of source examples: 6216

Subset created successfully.
Correct steps: 50
Error steps:   50
Total:         100
Saved to:      /content/cl-prm-project/data/prm_router/feasibility_100.jsonl

First example:
{
  "example_id": "step_contradiction_prm_test_p1_77",
  "source_row_index": 156,
  "classification": "step_contradiction",
  "question": "Solve for $x$: $\\frac{x}2 + \\frac{x}3 = 5$",
  "steps": [
    "Ok let's find a common denominator for these two fractions.",
    "That would be 6.",
    "Right. So now we have $\\frac{3x}6 + \\frac{2x}6 = 5$.",
    "If we add those two fractions together we get $\\frac{5x}6 = 5$.",
    "We'll then solve it incorrectly by dividing both sides by 5 first: $x/6 = 1

In [5]:
!python experiments/formal/prm_router/prepare_formal_splits.py \
  --config configs/experiments/prm_router_formal.json \
  --output-dir data/prm_router/formal

Loading dataset: hitsmy/PRMBench_Preview at revision 5cc7683d0ae5797f84d7aeac0607966f277c39e1
Source split: train
Source rows: 6216
Pilot examples: 100
Excluded pilot groups: 96
Available candidates after exclusion: 10815

train     :  600 examples, labels 0/1=300/300, groups=358
validation:  200 examples, labels 0/1=100/100, groups=113
test      :  400 examples, labels 0/1=200/200, groups=242

Leakage audit: passed
Manifest: /content/cl-prm-project/data/prm_router/formal/split_manifest.json


## Stage 2: Inference & Eval

- `dev` phase
- with `PathFinder-PRM-7B`

In [6]:
!python experiments/formal/prm_router/run_formal_inference.py \
  --phase development \
  --verifiers reasoneval pathfinder

Formal verifier-inference plan
Phase:       development
Splits:      train, validation
Verifiers:   reasoneval, pathfinder
Execute:     False

[1/4] /usr/bin/python3 experiments/feasibility/prm_router/run_disprm.py --input data/prm_router/formal/train.jsonl --output outputs/prm_router/formal/train/reasoneval.jsonl --model GAIR/ReasonEval-7B --revision 0a6556ef5c937bb17d265ba681b501fd60056cfe --warmup-examples 3 --resume --threshold 0.5
[2/4] /usr/bin/python3 experiments/feasibility/prm_router/run_disprm.py --input data/prm_router/formal/validation.jsonl --output outputs/prm_router/formal/validation/reasoneval.jsonl --model GAIR/ReasonEval-7B --revision 0a6556ef5c937bb17d265ba681b501fd60056cfe --warmup-examples 3 --resume --threshold 0.5
[3/4] /usr/bin/python3 experiments/feasibility/prm_router/run_pathfinder.py --input data/prm_router/formal/train.jsonl --output outputs/prm_router/formal/train/pathfinder.jsonl --model declare-lab/PathFinder-PRM-7B --revision 84a7412511836cb4ed74377d9c7

In [7]:
!python experiments/formal/prm_router/run_formal_inference.py \
  --phase development \
  --verifiers reasoneval pathfinder \
  --execute

Formal verifier-inference plan
Phase:       development
Splits:      train, validation
Verifiers:   reasoneval, pathfinder
Execute:     True

[1/4] /usr/bin/python3 experiments/feasibility/prm_router/run_disprm.py --input data/prm_router/formal/train.jsonl --output outputs/prm_router/formal/train/reasoneval.jsonl --model GAIR/ReasonEval-7B --revision 0a6556ef5c937bb17d265ba681b501fd60056cfe --warmup-examples 3 --resume --threshold 0.5
[2/4] /usr/bin/python3 experiments/feasibility/prm_router/run_disprm.py --input data/prm_router/formal/validation.jsonl --output outputs/prm_router/formal/validation/reasoneval.jsonl --model GAIR/ReasonEval-7B --revision 0a6556ef5c937bb17d265ba681b501fd60056cfe --warmup-examples 3 --resume --threshold 0.5
[3/4] /usr/bin/python3 experiments/feasibility/prm_router/run_pathfinder.py --input data/prm_router/formal/train.jsonl --output outputs/prm_router/formal/train/pathfinder.jsonl --model declare-lab/PathFinder-PRM-7B --revision 84a7412511836cb4ed74377d9c70

### Upload to HuggingFace

In [8]:
!hf upload cl-prm-team/cl-prm-artifacts \
  outputs/prm_router/formal/train \
  formal/train \
  --repo-type dataset

!hf upload cl-prm-team/cl-prm-artifacts \
  outputs/prm_router/formal/validation \
  formal/validation \
  --repo-type dataset

Start hashing 4 files.
Finished hashing 4 files.
https://huggingface.co/datasets/cl-prm-team/cl-prm-artifacts/tree/main/formal/train
Start hashing 4 files.
Finished hashing 4 files.
https://huggingface.co/datasets/cl-prm-team/cl-prm-artifacts/tree/main/formal/validation


### Evaluate `dev`

- freeze `reasoneval_threshold.json` and `development_selection.json`

In [9]:
!python experiments/formal/prm_router/select_reasoneval_threshold.py \
  --validation-reasoneval outputs/prm_router/formal/validation/reasoneval.jsonl

with open("outputs/prm_router/formal/validation/reasoneval_threshold.json", "r") as f:
    reasoneval_threshold = json.load(f)
    # threshold to be used for testing
    threshold = reasoneval_threshold["selected_threshold"]
    print(f"final tuned threshold: {threshold}")

ReasonEval threshold selection complete
Validation examples:        200
Selected threshold:         0.94696554
Balanced accuracy:          0.7250
Plain accuracy:             0.7250
Output:                     /content/cl-prm-project/outputs/prm_router/formal/validation/reasoneval_threshold.json
final tuned threshold: 0.9469655402936041


In [10]:
!python experiments/formal/prm_router/evaluate_development.py \
  --train-reasoneval outputs/prm_router/formal/train/reasoneval.jsonl \
  --train-pathfinder outputs/prm_router/formal/train/pathfinder.jsonl \
  --validation-reasoneval outputs/prm_router/formal/validation/reasoneval.jsonl \
  --validation-pathfinder outputs/prm_router/formal/validation/pathfinder.jsonl \
  --output outputs/prm_router/formal/pathfinder_development_selection.json \
  --reason-eval-threshold {threshold}

Formal development selection complete
Validation examples:       200
ReasonEval accuracy:       0.7250
PathFinder accuracy:       0.7500
Oracle accuracy:           0.9150
Expected-gain @ 20%:       0.7700
Selected cost-aware:      lambda_h=1.5, mu=0.5, accuracy=0.7800
Output:                    outputs/prm_router/formal/pathfinder_development_selection.json


In [11]:
!hf upload cl-prm-team/cl-prm-artifacts \
  outputs/prm_router/formal/validation/reasoneval_threshold.json \
  evaluation/router/reasoneval_threshold.json \
  --repo-type dataset

!hf upload cl-prm-team/cl-prm-artifacts \
  outputs/prm_router/formal/pathfinder_development_selection.json \
  evaluation/router/pathfinder_development_selection.json \
  --repo-type dataset

Removing 1 file(s) from commit that have not changed.
No files have been modified since last commit. Skipping to prevent empty commit.
https://huggingface.co/datasets/cl-prm-team/cl-prm-artifacts/blob/main/evaluation/router/reasoneval_threshold.json
https://huggingface.co/datasets/cl-prm-team/cl-prm-artifacts/blob/main/evaluation/router/pathfinder_development_selection.json


## Stage 3: Inference & Eval

- `test` phase
- with `PathFinder-PRM-7B`
- ensure protocol is frozen

In [12]:
!python experiments/formal/prm_router/run_formal_inference.py \
  --phase test \
  --verifiers reasoneval pathfinder \
  --reason-eval-threshold {threshold} \
  --confirm-test-protocol-frozen

Formal verifier-inference plan
Phase:       test
Splits:      test
Verifiers:   reasoneval, pathfinder
Execute:     False

[1/2] /usr/bin/python3 experiments/feasibility/prm_router/run_disprm.py --input data/prm_router/formal/test.jsonl --output outputs/prm_router/formal/test/reasoneval.jsonl --model GAIR/ReasonEval-7B --revision 0a6556ef5c937bb17d265ba681b501fd60056cfe --warmup-examples 3 --resume --threshold 0.9469655402936041
[2/2] /usr/bin/python3 experiments/feasibility/prm_router/run_pathfinder.py --input data/prm_router/formal/test.jsonl --output outputs/prm_router/formal/test/pathfinder.jsonl --model declare-lab/PathFinder-PRM-7B --revision 84a7412511836cb4ed74377d9c703eb5638d814c --warmup-examples 3 --resume --threshold 0.5 --max-input-tokens 4096 --attention-implementation flash_attention_2

Plan only; no model inference was started.
Add --execute to run these commands sequentially.


In [13]:
!python experiments/formal/prm_router/run_formal_inference.py \
  --phase test \
  --verifiers reasoneval pathfinder \
  --reason-eval-threshold {threshold} \
  --confirm-test-protocol-frozen \
  --execute

Formal verifier-inference plan
Phase:       test
Splits:      test
Verifiers:   reasoneval, pathfinder
Execute:     True

[1/2] /usr/bin/python3 experiments/feasibility/prm_router/run_disprm.py --input data/prm_router/formal/test.jsonl --output outputs/prm_router/formal/test/reasoneval.jsonl --model GAIR/ReasonEval-7B --revision 0a6556ef5c937bb17d265ba681b501fd60056cfe --warmup-examples 3 --resume --threshold 0.9469655402936041
[2/2] /usr/bin/python3 experiments/feasibility/prm_router/run_pathfinder.py --input data/prm_router/formal/test.jsonl --output outputs/prm_router/formal/test/pathfinder.jsonl --model declare-lab/PathFinder-PRM-7B --revision 84a7412511836cb4ed74377d9c703eb5638d814c --warmup-examples 3 --resume --threshold 0.5 --max-input-tokens 4096 --attention-implementation flash_attention_2

Running command 1/2
2026-09-23 12:02:11.589028: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point 

In [14]:
!hf upload cl-prm-team/cl-prm-artifacts \
  outputs/prm_router/formal/test \
  formal/test \
  --repo-type dataset

Start hashing 4 files.
Finished hashing 4 files.
https://huggingface.co/datasets/cl-prm-team/cl-prm-artifacts/tree/main/formal/test


### Evaluate `test`

In [15]:
!python experiments/formal/prm_router/evaluate_test.py \
  --train-reasoneval outputs/prm_router/formal/train/reasoneval.jsonl \
  --train-pathfinder outputs/prm_router/formal/train/pathfinder.jsonl \
  --test-reasoneval outputs/prm_router/formal/test/reasoneval.jsonl \
  --test-pathfinder outputs/prm_router/formal/test/pathfinder.jsonl \
  --development-selection outputs/prm_router/formal/pathfinder_development_selection.json \
  --confirm-protocol-frozen

Frozen held-out PRM Router evaluation complete
Test examples:              400
ReasonEval:                 0.7175
PathFinder:                 0.7900
Random @ 20%:               0.7450
Low score @ 20%:            0.7000
Uncertainty @ 20%:          0.7450
Failure prediction @ 20%:   0.8075
Benefit only @ 20%:         0.8025
Expected gain @ 20%:        0.8175
Cost-aware @ 20%:           0.8175
Oracle:                     0.9225
Output:                     /content/cl-prm-project/outputs/prm_router/formal/test/formal_test_results.json


In [16]:
!hf upload cl-prm-team/cl-prm-artifacts \
  outputs/prm_router/formal/test/formal_test_results.json \
  evaluation/router/formal_test_results.json \
  --repo-type dataset

https://huggingface.co/datasets/cl-prm-team/cl-prm-artifacts/blob/main/evaluation/router/formal_test_results.json
